# 02. Indexing, Slicing & Boolean Masking: Beginner Guide

### 🌟 What is Array Indexing, Slicing & Boolean Masking in NumPy?
Extracting specific subsets of data quickly is critical in data science. NumPy enables multi-dimensional slicing without copying memory (zero-copy views), as well as **Boolean Masking**—filtering millions of numbers matching conditions (like `amounts > 1000`) in a single lightning-fast step.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Standard & Multi-Axis Slicing**: Slicing along row and column dimensions in $O(1)$ time.
- **View Checking (`.base`)**: Distinguishing memory-sharing views from independent memory owners.
- **Boolean Masking**: Filtering array elements via vectorized condition masks without Python loops.
- **Fancy Indexing**: Extracting non-contiguous elements using integer array coordinates (which always creates a copy).


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 1D Slicing: `arr[start:stop:step]`
Extracts a sub-sequence of transaction amounts. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

**Syntax:** `amounts[0:10:2]`


In [2]:
print('1D Sliced Amounts (first 5 even indices):', amounts[0:10:2])

1D Sliced Amounts (first 5 even indices): [1216.33  136.66 1284.68  781.65  679.9 ]


### 🔹 Multi-Axis 2D Slicing
Slices rows and columns from a multi-feature transaction matrix. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

**Syntax:** `tx_matrix[0:5, 0:2]`


In [3]:
tx_matrix = np.column_stack([amounts[:100], account_ages[:100]])
print('2D Sliced Matrix (Rows 0-3, Cols 0-1):\n', tx_matrix[0:3, 0:2])

2D Sliced Matrix (Rows 0-3, Cols 0-1):
 [[1216.33   56.  ]
 [ 324.99  112.  ]
 [ 136.66   68.  ]]


### 🔹 View Checking with `.base`
Proves basic slice shares memory with original transaction matrix. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `slice_view.base is not None`


In [4]:
slice_view = tx_matrix[0:5, 0]
copy_view = tx_matrix[0:5, 0].copy()
print('slice_view shares memory (is view)?:', slice_view.base is not None)
print('copy_view is independent copy?:', copy_view.base is None)

slice_view shares memory (is view)?: True
copy_view is independent copy?: True


### 🔹 Vectorized Boolean Masking
Filters all fraudulent high-value transactions (> $500). NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

**Syntax:** `amounts[(amounts > 500) & (fraud_flags == 1)]`


In [5]:
fraud_mask = (fraud_flags == 1) & (amounts > 500.0)
high_val_fraud = amounts[fraud_mask]
print(f'High-Value Fraud Amounts Found ({len(high_val_fraud)} txs):', high_val_fraud[:5].round(2))

High-Value Fraud Amounts Found (1540 txs): [1805.16  927.5  1893.8  1944.85 1593.81]


### 🔹 Fancy Indexing with Integer Arrays
Extracts transactions at specific non-contiguous index locations into a deep copy. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

**Syntax:** `amounts[[0, 42, 100, 999]]`


In [6]:
fancy_sample = amounts[[0, 42, 100, 500]]
print('Fancy Indexed Amounts:', fancy_sample)
print('Is Fancy Indexing an isolated copy?:', fancy_sample.base is None)

Fancy Indexed Amounts: [1216.33  895.09  698.29 1431.63]
Is Fancy Indexing an isolated copy?: True


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: In-Place Outlier Clamping via Boolean Masking

**Approach:** Clamp all transaction amounts exceeding $2000 down to $2000 in-place without memory allocation.
**Syntax:** `amounts[amounts > 2000.0] = 2000.0`


In [7]:
clamped_amounts = amounts.copy()
clamped_amounts[clamped_amounts > 2000.0] = 2000.0
print('Max Amount After In-Place Clamping:', clamped_amounts.max())

Max Amount After In-Place Clamping: 1999.98
